# Island Duel: starter workflow

Run this notebook from the extracted participant folder. It demonstrates the
observation and action contract, a small baseline, saved gameplay and the full
evaluation command. The baseline is illustrative; it is not a qualifying solution.

Requires Python 3.10+ and the supplied engine for your operating system. No
third-party Python library is needed by these cells. A notebook viewer is optional.


In [ ]:
import json
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
assert (ROOT / "client.py").is_file(), "Open the notebook in the participant folder"
OUTPUT = ROOT / "baseline-demo"
ordinal = 1
while OUTPUT.exists():
    OUTPUT = ROOT / f"baseline-demo-{ordinal}"
    ordinal += 1
OUTPUT.mkdir()
print("Evidence folder:", OUTPUT)


## Inspect a reproducible practice observation

Each state contains public information and a mask of legal actions. Hidden
terrain is represented by `None`. Read `task_description.md` for the rules.


In [ ]:
from client import ArenaClient

with ArenaClient() as env:
    response = env.reset(0, seed=144)
    state = response["observation"]
    print("Observation fields:", sorted(state))
    print("Legal actions:", state["legal_actions"][:8])
    print("Both players' scores:", state["scores"])


## Write a baseline agent

The runner imports `choose_action(state)`. Return an integer from
`state["legal_actions"]`, or `None` to rest. This policy compares immediate
territory value with energy cost. It does not model hidden terrain, future
frontiers or the opponent. Improve it or replace it with your own policy.


In [ ]:
agent_path = OUTPUT / "baseline_agent.py"
agent_path.write_text('''"""A simple immediate-value baseline; no training or hidden information."""

def choose_action(state):
    seat = state["seat"]
    def value(action):
        if action is None:
            return min(3, 9 - state["energy"][seat]) * 1.5
        capture = state["owners"][action] == 1 - seat
        gain = state["weights"][action] * (2 if capture else 1)
        cost = state["costs"][action] + (2 if capture else 0)
        return gain - 1.5 * cost
    return max(state["legal_actions"], key=value)
''', encoding="utf-8")


## Play and save a complete practice game

The command writes a JSON record and a standalone HTML replay. A loss is a valid
baseline result. Open the HTML file to inspect the decisions and visible terrain.


In [ ]:
practice = OUTPUT / "practice.json"
subprocess.run([sys.executable, "run_agent.py", "--agent", str(agent_path),
                "--arena", "0", "--seed", "144", "--output", str(practice)], check=True)
game = json.loads(practice.read_text(encoding="utf-8"))["games"][0]
assert len(game["participant_actions"]) == 128
print("Final scores:", game["scores"])
print("Replay:", practice.with_suffix(".html"))


## Run a full evaluation when ready

Set the switch below to `True` to evaluate this baseline, or change `agent_path`
to your improved agent first. All 512 games must finish. Qualification requires
at least 308 wins and wins in at least 24 groups. The runner preserves outcomes
and writes representative replays; it releases a chart only after qualification.
This cell does not supply a puzzle solution. Use a fresh output name for each run.


In [ ]:
RUN_FULL_EVALUATION = False
if RUN_FULL_EVALUATION:
    evaluation = OUTPUT / "evaluation.json"
    subprocess.run([sys.executable, "evaluate.py", "--agent", str(agent_path),
                    "--output", str(evaluation)], check=True)
    summary = json.loads(evaluation.read_text(encoding="utf-8"))["summary"]
    print({key: summary[key] for key in ["complete", "wins", "winning_groups", "qualified"]})
else:
    print("Full evaluation is optional in this starter demonstration.")


## Submission

Recover the flag from the chart earned by your qualifying agent. Submit the flag
and an explanation URL containing source, dependencies, commands, every evaluation
attempt and gameplay evidence. Record total elapsed time through flag recovery.
